In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, f1_score

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
EPOCHS = 50
PATIENCE = 10

In [ ]:
data_dir = "Datasets/RQD"

# ImageNet normalization
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

# Data augmentation referred to Lank and Friedjungova (2022)
data_transforms = {

    # data augmentation 
    'train': transforms.Compose([
        transforms.Resize((360, 360)),
        transforms.ColorJitter(contrast=0.2),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),

    'val': transforms.Compose([
        transforms.Resize((360, 360)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),

    'test': transforms.Compose([
        transforms.Resize((360, 360)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
}


In [ ]:
image_datasets = {
    x: datasets.ImageFolder(
        root=f"{data_dir}/{x}",
        transform=data_transforms[x]
    )
    for x in ['train', 'val', 'test']
}


dataloaders = {
    'train': DataLoader(
        image_datasets['train'],
        batch_size=32,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    ),

    'val': DataLoader(
        image_datasets['val'],
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    ),

    'test': DataLoader(
        image_datasets['test'],
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
}


class_names = image_datasets['train'].classes
num_classes = len(class_names)

print("Classes:", class_names)

In [ ]:
# # MobileNet_V2

weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
model = models.mobilenet_v2(weights=weights)

In [ ]:
# Freeze all
for param in model.features.parameters():
    param.requires_grad = False

# Change classifier
num_ftrs = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    num_ftrs,
    num_classes
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

In [ ]:
def train_model(
    model,
    optimizer,
    phase_name,
    num_epochs,
    patience
):

    history = defaultdict(list)

    best_val_acc = 0.0
    best_epoch = 0
    epochs_without_improvement = 0

    best_model_state = copy.deepcopy(model.state_dict())


    print(f"\n{'=' * 60}")
    print(f"{phase_name}")
    print(f"{'=' * 60}")


    for epoch in range(num_epochs):

        print(f"\nEpoch {epoch + 1}/{num_epochs}")


        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        model.train()

        train_loss_sum = 0.0
        train_corrects = 0
        train_count = 0


        for inputs, labels in dataloaders['train']:

            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)

            loss.backward()

            optimizer.step()


            batch_size = inputs.size(0)

            train_loss_sum += loss.item() * batch_size

            train_corrects += (
                torch.sum(preds == labels).item()
            )

            train_count += batch_size


        train_loss = train_loss_sum / train_count

        train_acc = train_corrects / train_count


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        model.eval()

        val_loss_sum = 0.0
        val_corrects = 0
        val_count = 0


        with torch.no_grad():

            for inputs, labels in dataloaders['val']:

                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)

                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)


                batch_size = inputs.size(0)

                val_loss_sum += loss.item() * batch_size

                val_corrects += (
                    torch.sum(preds == labels).item()
                )

                val_count += batch_size


        val_loss = val_loss_sum / val_count

        val_acc = val_corrects / val_count


        # Store history
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)

        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)


        print(
            f"Train: loss={train_loss:.4f} "
            f"acc={train_acc:.4f} | "
            f"Val: loss={val_loss:.4f} "
            f"acc={val_acc:.4f}"
        )


        if val_acc > best_val_acc:

            best_val_acc = val_acc

            best_epoch = epoch + 1

            best_model_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

            print(
                f"New best validation accuracy: "
                f"{best_val_acc:.4f}"
            )

        else:

            epochs_without_improvement += 1

            print(
                f"No improvement for "
                f"{epochs_without_improvement}/"
                f"{patience} epochs"
            )


        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if epochs_without_improvement >= patience:

            print("\nEarly stopping triggered.")

            break


    model.load_state_dict(best_model_state)


    print(
        f"\nBest {phase_name} validation accuracy: "
        f"{best_val_acc:.4f} "
        f"(epoch {best_epoch})"
    )


    return (
        model,
        history,
        best_val_acc,
        best_epoch
    )

In [ ]:
# Phase 1: Freeze backbone

model, history_phase1, phase1_best_acc, phase1_best_epoch = train_model(
    model=model,
    optimizer=optimizer,
    phase_name="PHASE 1 - Frozen MobileNetV2 backbone",
    num_epochs=EPOCHS,
    patience=PATIENCE
)

In [ ]:
# Phase 2: Unfreeze all

for param in model.parameters():
    param.requires_grad = True


optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)


model, history_phase2, phase2_best_acc, phase2_best_epoch = train_model(
    model=model,
    optimizer=optimizer,
    phase_name="PHASE 2 - Fine-tuning full MobileNetV2",
    num_epochs=EPOCHS,
    patience=PATIENCE
)

In [ ]:
model_name = model.__class__.__name__

best_path = f"ModelTraining/best_{model_name}_RQD.pt"

Path("ModelTraining").mkdir(
    parents=True,
    exist_ok=True
)

torch.save(
    {
        "model_state": model.state_dict(),
        "class_names": class_names,
        "phase1_best_val_acc": phase1_best_acc,
        "phase1_best_epoch": phase1_best_epoch,
        "phase2_best_val_acc": phase2_best_acc,
        "phase2_best_epoch": phase2_best_epoch
    },
    best_path
)


print(f"\nModel saved to: {best_path}")

In [ ]:
model.eval()

test_loss_sum = 0.0
test_corrects = 0
test_count = 0

all_labels = []
all_preds = []


with torch.no_grad():

    for inputs, labels in dataloaders['test']:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        _, preds = torch.max(outputs, 1)


        batch_size = inputs.size(0)

        test_loss_sum += loss.item() * batch_size

        test_corrects += (
            torch.sum(preds == labels).item()
        )

        test_count += batch_size


        all_labels.extend(
            labels.cpu().numpy()
        )

        all_preds.extend(
            preds.cpu().numpy()
        )


test_loss = test_loss_sum / test_count

test_acc = test_corrects / test_count


print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
print("\nClassification Report:\n")

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        digits=4
    )
)


f1_per_class = f1_score(
    all_labels,
    all_preds,
    average=None
)


print("\nF1-score per class:")

for class_name, score in zip(
    class_names,
    f1_per_class
):

    print(
        f"{class_name}: "
        f"{score:.4f} "
        f"({score * 100:.2f}%)"
    )

In [ ]:
# Phase 1 + Phase 2
train_acc_history = (
    history_phase1["train_acc"] +
    history_phase2["train_acc"]
)

val_acc_history = (
    history_phase1["val_acc"] +
    history_phase2["val_acc"]
)

train_loss_history = (
    history_phase1["train_loss"] +
    history_phase2["train_loss"]
)

val_loss_history = (
    history_phase1["val_loss"] +
    history_phase2["val_loss"]
)

phase1_length = len(history_phase1["train_acc"])

epochs = range(1, len(train_acc_history) + 1)


# ACC PLOT
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    train_acc_history,
    label="Train Accuracy"
)

plt.plot(
    epochs,
    val_acc_history,
    label="Validation Accuracy"
)

plt.axvline(
    phase1_length,
    linestyle="--",
    label="Start Fine-tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("MobileNetV2 Accuracy")

# Start x-axis at 0
plt.xlim(left=0)

plt.legend()
plt.tight_layout()
plt.show()


# LOSS PLOT
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    train_loss_history,
    label="Train Loss"
)

plt.plot(
    epochs,
    val_loss_history,
    label="Validation Loss"
)

plt.axvline(
    phase1_length,
    linestyle="--",
    label="Start Fine-tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MobileNetV2 Loss")

plt.xlim(left=0)

plt.legend()
plt.tight_layout()
plt.show()